# Imports, Options, and Functions

In [8]:
import requests
import traceback
import hashlib
import json
import os
import re
import pandas as pd

superdirec = "/home/easlinger/data"
# studies = ["OSD-927", "OSD-928", "OSD-929", "OSD-930", "OSD-931", "OSD-932"]
studies = ["OSD-923", "OSD-934"]
overwrite = False
cores, mem = 32, 64  # cpu cores/RAM to allocate for CellRanger


def get_osd_samples(study_id, pattern=None, field="REST_URL"):
    """Retrieve metadata and file names for an OSDR study."""
    url = str("https://visualization.osdr.nasa.gov/biodata/api/v2/dataset/"
              f"{study_id}/files")
    response = requests.get(url)
    samples = response.json()
    samples = dict([(x, samples[study_id]["files"][x][
        field]) for x in samples[study_id]["files"]])
    if pattern is not None:
        samples = {k: samples[k] for k in samples if (pattern in k)}
    return samples


def compute_md5(filepath, chunk_size=8192):
    """Compute MD5 checksum of a local file."""
    md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            md5.update(chunk)
    return md5.hexdigest()


def get_and_write_osd_data(rest_url, file_name, file_out,
                           study_id=None, unzip=False, overwrite=False):
    """Retrieve OSD file and write to disk."""
    if overwrite is False and os.path.exists(file_out):
        print(f"{os.path.join(out_dir, file_name)} already exists")
    else:
        meta = requests.get(rest_url).json()
        if study_id is None:
            if len(meta) == 1:
                study_id = list(meta.keys())[0]
            else:
                raise ValueError(f"Multiple studies ({meta.keys()}) found. "
                                 "Please specify `study_id`.")
        download_url = meta[study_id]["files"][file_name]["URL"]
        response = requests.get(download_url, stream=True)
        with open(file_out, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        if unzip is True:
            os.system(f"unzip -o {file_out} -d {os.path.dirname(file_out)}")
        print(f"Wrote study {study_id} {file_name} to {file_out}.")


def get_reference_genome(genome, release, species, out_dir,
                         cellranger_ref_dir=None, overwrite=False):
    """
    Download reference genome and create CellRanger reference.

    Set `cellranger_ref_dir=False` to skip CellRanger reference
    creation and just download/unzip
    """
    # Construct Reference Genome URLs and File Paths
    uref = str(f"https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_"
               f"{species}/release_{release}")
    ref1 = os.path.join(out_dir, f"{genome}.primary_assembly.genome.fa")
    ref2 = os.path.join(out_dir, f"gencode.v{release}.annotation.gtf")
    dir_ref = os.path.join(out_dir, f"mkref_{genome}") if (
        cellranger_ref_dir is None) else cellranger_ref_dir  # for CellRanger
    os.makedirs(out_dir, exist_ok=True)  # make reference genome directory
    # Download/Unzip Reference Genome Files If Needed
    chksum = None  # not retrieved yet; will get if needed
    for x in [ref1, ref2]:
        # Download Reference File If Needed
        if overwrite is True or (os.path.exists(x + ".gz") is False and (
                os.path.exists(x) is False)):  # get file if needed
            os.system(f"wget -P {out_dir} {uref}/{os.path.basename(x)}.gz")
            # Check Checksums
            try:
                if chksum is None:
                    res = requests.get(str(
                        "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_"
                        f"{species}/release_{release}/MD5SUMS"))  # download
                    chksum = {line.split()[1]: line.split()[0] for line in [
                        v.strip("\r").strip()
                        for v in res.text.split("\n") if v]}  # parse lines
                computed_md5 = compute_md5(x + ".gz")  # compute hash
                actual = chksum.get(os.path.basename(x + ".gz"))  # expected
            except Exception as err:
                print(f"Failed to verify checksum for {x}: {err}")
                traceback.print_exc()
                computed_md5, actual = None, None
            if computed_md5 is not None and computed_md5 != actual:
                raise ValueError(f"MD5 checksum mismatch for {x}.gz")
        # Unzip Reference File If Needed
        if overwrite is True or os.path.exists(os.path.join(
                out_dir, x)) is False:
            os.system(f"gunzip {os.path.join(out_dir, x)}.gz")  # unzip
    # Create CellRanger Reference (If Needed)
    if cellranger_ref_dir is not False and os.path.exists(dir_ref) is False:
        os.system(f"cellranger mkref --genome={genome}"
                  f" --output-dir={dir_ref}"
                  f" --fasta={ref1} --genes={ref2}")
    else:
        if cellranger_ref_dir is not False:
            print(f"\n\n\nReference genome already exists at {dir_ref}.\n\n")
    return dir_ref

# Download Data

In [7]:
# Download FASTQs
samples = {k: get_osd_samples(k, pattern="fastq") for k in studies}
for study_id in studies:
    out_dir = os.path.join(superdirec, f"{study_id}_raw")
    os.makedirs(out_dir, exist_ok=True)
    for file_name, rest_url in samples[study_id].items():
        try:
            get_and_write_osd_data(
                rest_url, file_name, os.path.join(out_dir, file_name),
                study_id=study_id, unzip=False, overwrite=overwrite)
        except Exception as e:
            print(f"Failed to download {file_name}: {e}")
            traceback.print_exc()

# Download Metadata
samples_metadata = {x: get_osd_samples(x, "metadata") for x in studies}
for study_id in studies:
    out_dir = os.path.join(superdirec, f"{study_id}_metadata_{study_id}-ISA")
    os.makedirs(out_dir, exist_ok=True)
    for file_name, rest_url in samples_metadata[study_id].items():
        get_and_write_osd_data(
            rest_url, file_name, os.path.join(out_dir, file_name),
            study_id=study_id, unzip=False, overwrite=overwrite)

Wrote study OSD-923 GLDS-751_scRNA-Seq_SRX28491892_R2_raw.fastq.gz to /home/easlinger/data/OSD-923_raw/GLDS-751_scRNA-Seq_SRX28491892_R2_raw.fastq.gz.
Wrote study OSD-923 GLDS-751_scRNA-Seq_SRX28491892_R1_raw.fastq.gz to /home/easlinger/data/OSD-923_raw/GLDS-751_scRNA-Seq_SRX28491892_R1_raw.fastq.gz.
Wrote study OSD-923 GLDS-751_scRNA-Seq_SRX28491893_R2_raw.fastq.gz to /home/easlinger/data/OSD-923_raw/GLDS-751_scRNA-Seq_SRX28491893_R2_raw.fastq.gz.
Wrote study OSD-923 GLDS-751_scRNA-Seq_SRX28491893_R1_raw.fastq.gz to /home/easlinger/data/OSD-923_raw/GLDS-751_scRNA-Seq_SRX28491893_R1_raw.fastq.gz.
Wrote study OSD-923 GLDS-751_scRNA-Seq_SRX28491894_R2_raw.fastq.gz to /home/easlinger/data/OSD-923_raw/GLDS-751_scRNA-Seq_SRX28491894_R2_raw.fastq.gz.
Wrote study OSD-923 GLDS-751_scRNA-Seq_SRX28491894_R1_raw.fastq.gz to /home/easlinger/data/OSD-923_raw/GLDS-751_scRNA-Seq_SRX28491894_R1_raw.fastq.gz.
Wrote study OSD-923 GLDS-751_scRNA-Seq_SRX28491953_R2_raw.fastq.gz to /home/easlinger/data/OSD

# Download and Prepare Reference Genome

In [9]:
# Options
genome, release, species = "GRCm39", "M36", "mouse"
# genome, release, species = "GRCh38", "49", "human"  # human example
dref = os.path.join(superdirec, "reference_genomes",
                    f"refdata-{genome}-{release}-{species}")

# Get & Prepare Reference Genome
dir_ref = get_reference_genome(genome, release, species, dref,
                               cellranger_ref_dir=None, overwrite=overwrite)




Reference genome already exists at /home/easlinger/data/reference_genomes/refdata-GRCm39-M36-mouse/mkref_GRCm39.




# Verify File Integrity

Check checksum files to make sure downloaded fastq files are intact.

In [10]:
chksums = {k: get_osd_samples(k, pattern="md5", field="URL") for k in studies}
for study_id in studies:  # loop OSD studies
    out_dir = os.path.join(superdirec, f"{study_id}_raw")
    if len(chksums[study_id].keys()) > 1:
        raise ValueError(f"Multiple md5 checksum files found for {study_id}")
    res = requests.get(chksums[study_id][list(chksums[study_id].keys())[0]])
    lines = [v.strip("\r").strip() for v in res.text.split("\n") if v]
    chksum_dict = {line.split()[1]: line.split()[0] for line in lines}
    for fname, rest_url in samples[study_id].items():  # loop files in study
        if os.path.exists(os.path.join(out_dir, fname)):
            chk_f = [u for u in chksum_dict if fname.endswith(u)]
            if len(chk_f) > 1 or len(chk_f) == 0:
                raise ValueError(f"{'>1' if len(chk_f) > 0 else 'No'} "
                                 f"{fname} checksums found: {chk_f}")
            expected = chksum_dict[chk_f[0]]  # expected checksum from OSDR
            actual = compute_md5(os.path.join(out_dir, fname))  # and file's
            if actual == expected:  # checksums match?
                print(f"[PASS] {fname} checksum matches. {actual}={expected}")
            else:
                raise ValueError(f"[FAIL] {study_id} {fname}: checksum "
                                 f"expected={expected}, got={actual}")
        else:
            print(f"{os.path.join(out_dir, fname)} isn't downloaded.")

[PASS] GLDS-751_scRNA-Seq_SRX28491892_R2_raw.fastq.gz checksum matches. bcfddeefc954b84c532ad5b4954e3790=bcfddeefc954b84c532ad5b4954e3790
[PASS] GLDS-751_scRNA-Seq_SRX28491892_R1_raw.fastq.gz checksum matches. 49a93280343792006c7d9bd2b646baf3=49a93280343792006c7d9bd2b646baf3
[PASS] GLDS-751_scRNA-Seq_SRX28491893_R2_raw.fastq.gz checksum matches. 6e359247d6f6be4ea03aa79fb322abeb=6e359247d6f6be4ea03aa79fb322abeb
[PASS] GLDS-751_scRNA-Seq_SRX28491893_R1_raw.fastq.gz checksum matches. 4dff5b57ecbf4cb6b4d38198b4fa28e7=4dff5b57ecbf4cb6b4d38198b4fa28e7
[PASS] GLDS-751_scRNA-Seq_SRX28491894_R2_raw.fastq.gz checksum matches. 0c4c39a1e137ebf4284f429727e778b2=0c4c39a1e137ebf4284f429727e778b2
[PASS] GLDS-751_scRNA-Seq_SRX28491894_R1_raw.fastq.gz checksum matches. 69d49b3048bafa5453d7c976dd269b7b=69d49b3048bafa5453d7c976dd269b7b
[PASS] GLDS-751_scRNA-Seq_SRX28491953_R2_raw.fastq.gz checksum matches. 9cccfe9842fc0ca8b802f818cfc48c8c=9cccfe9842fc0ca8b802f818cfc48c8c
[PASS] GLDS-751_scRNA-Seq_SRX28491

# Prepare for CellRanger Run

## Create Symlinks

If files are not named according to CellRanger convention, make symlinks that do follow it. 

If you don't need the symlinks, just run the following instead of this code cell:

```
samples_n = {k: {} for k in samples}  # to hold symlinks
for study_id in samples:  # iterate OSDR studies
    out_dir = os.path.join(superdirec, f"{study_id}_raw")
    for f in samples[study_id]:  # iterate files in the study
        samples_n[study_id][os.path.join(out_dir, f)] = os.path.join(
            out_dir, f)
sample_map = pd.concat([pd.Series(
    samples_n[x]) for x in samples_n], keys=samples_n.keys(), names=[
        "Study", "Original"]).reset_index(1)
sample_map = sample_map.assign(Sample=sample_map.apply(
    lambda x: os.path.basename(x.iloc[1]).split("_")[0], axis=1)).set_index(
        "Sample", append=True).rename({0: "Symlink"}, axis=1)
```

**Make sure that the code** `os.path.basename(x.iloc[1]).split("_")[0]` (where `x` is the original file name) **actually properly extracts the sample name.** This code assumes the file is in the format `<sample_id (without any underscores in it)>_<rest of file name>`.

In [15]:
# Create CellRanger-Friendly Symlinks
samples_n = {k: {} for k in samples}  # to hold symlinks
for study_id in samples:  # iterate OSDR studies
    out_dir = os.path.join(superdirec, f"{study_id}_raw")
    for f in samples[study_id]:  # iterate files in the study
        srx, read = re.search(r"(SRX\d+)_(R[12])_raw\.fastq\.gz$", f).groups()
        samples_n[study_id][os.path.join(out_dir, f)] = os.path.join(
            out_dir, f"{srx}_S1_L001_{read}_001.fastq.gz")  # symlink path
        if not os.path.exists(samples_n[study_id][os.path.join(out_dir, f)]):
            os.symlink(os.path.join(out_dir, f), samples_n[study_id][
                os.path.join(out_dir, f)])  # CellRanger-friendly symlink

# Check Symlink Mapping
sample_map = pd.concat([pd.Series(
    samples_n[x]) for x in samples_n], keys=samples_n.keys(), names=[
        "Study", "Original"]).reset_index(1)
sample_map = sample_map.assign(Sample=sample_map.apply(
    lambda x: os.path.basename(x.iloc[1]).split("_")[0], axis=1)).set_index(
        "Sample", append=True).rename({0: "Symlink"}, axis=1).sort_index()
print(sample_map)
map_dir = sample_map.map(os.path.dirname)
assert all(map_dir.iloc[:, 0] == map_dir.iloc[:, 1])  # all directories =?
map_base = sample_map.map(os.path.basename).map(lambda x: (
    x.split("Seq_")[1] if "Seq_" in x else x).split(".fastq.gz")[0]).map(
        lambda x: x.split("_raw")[0].split("_001")[0])  # key file name parts
map_rep = map_base.map(lambda x: x.split("_")[-1])
assert all(map_rep.iloc[:, 0] == map_rep.iloc[:, 1])  # replicate suffixes =?
map_sub = map_base.map(lambda x: x.split("_")[0])
assert all(map_sub.iloc[:, 0] == map_sub.iloc[:, 1])  # sample prefixes =?

                                                              Original  \
Study   Sample                                                           
OSD-923 SRX28491892  /home/easlinger/data/OSD-923_raw/GLDS-751_scRN...   
        SRX28491892  /home/easlinger/data/OSD-923_raw/GLDS-751_scRN...   
        SRX28491893  /home/easlinger/data/OSD-923_raw/GLDS-751_scRN...   
        SRX28491893  /home/easlinger/data/OSD-923_raw/GLDS-751_scRN...   
        SRX28491894  /home/easlinger/data/OSD-923_raw/GLDS-751_scRN...   
        SRX28491894  /home/easlinger/data/OSD-923_raw/GLDS-751_scRN...   
        SRX28491953  /home/easlinger/data/OSD-923_raw/GLDS-751_scRN...   
        SRX28491953  /home/easlinger/data/OSD-923_raw/GLDS-751_scRN...   
        SRX28491954  /home/easlinger/data/OSD-923_raw/GLDS-751_scRN...   
        SRX28491954  /home/easlinger/data/OSD-923_raw/GLDS-751_scRN...   
        SRX28491955  /home/easlinger/data/OSD-923_raw/GLDS-751_scRN...   
        SRX28491955  /home/easlinger/d

## Create CellRanger Commands

You can then copy-paste into your terminal or use the sample_map object to write code to loop through and run the CellRanger commands.

For separate scripts by study ID:

```
for q in sample_map.index.get_level_values("Study").unique():
    cmnds = sample_map.loc[q]["CellRanger_Command"]
    print(f"\n\n\n{'=' * 80}\n{q} Bash Code\n{'=' * 80}\n\n#!/bin/bash\n")
    print("\n".join([f"{c}  # {f}\n" for f, c in cmnds.items()]))
```

In [16]:
for q in sample_map.index.get_level_values("Study").unique():
    out_dir_new = os.path.join(superdirec, q)
    for i in sample_map.loc[q].index.get_level_values("Sample").unique():
        cmd = f"cellranger count --create-bam=false"  # command base syntax
        cmd += f" --id={i} --sample={i}"  # add sample ID arguments
        cmd += f" --localcores={cores} --localmem={mem}"  # compute resources
        cmd += f" --transcriptome={dir_ref}"  # genome reference path
        cmd += f" --fastqs={out_dir_new}_raw --output-dir={out_dir_new}"
        sample_map.loc[(q, i), "CellRanger_Command"] = cmd
# for q in sample_map.index.get_level_values("Study").unique():
#     print(f"\n\n\n{'=' * 80}\n{q}\n{'=' * 80}\n\n")
#     print("\n".join(sample_map.loc[q]["CellRanger_Command"].to_list()))

# Bash Code
cmnds = sample_map["CellRanger_Command"]
print(f"\n\n\n{'=' * 80}\nBash Code\n{'=' * 80}\n\n#!/bin/bash\n")
print("\n".join([f"{c}  # {f}\n" for f, c in cmnds.items()]))




Bash Code

#!/bin/bash

cellranger count --create-bam=false --id=SRX28491892 --sample=SRX28491892 --localcores=32 --localmem=64 --transcriptome=/home/easlinger/data/reference_genomes/refdata-GRCm39-M36-mouse/mkref_GRCm39 --fastqs=/home/easlinger/data/OSD-923_raw --output-dir=/home/easlinger/data/OSD-923  # ('OSD-923', 'SRX28491892')

cellranger count --create-bam=false --id=SRX28491892 --sample=SRX28491892 --localcores=32 --localmem=64 --transcriptome=/home/easlinger/data/reference_genomes/refdata-GRCm39-M36-mouse/mkref_GRCm39 --fastqs=/home/easlinger/data/OSD-923_raw --output-dir=/home/easlinger/data/OSD-923  # ('OSD-923', 'SRX28491892')

cellranger count --create-bam=false --id=SRX28491893 --sample=SRX28491893 --localcores=32 --localmem=64 --transcriptome=/home/easlinger/data/reference_genomes/refdata-GRCm39-M36-mouse/mkref_GRCm39 --fastqs=/home/easlinger/data/OSD-923_raw --output-dir=/home/easlinger/data/OSD-923  # ('OSD-923', 'SRX28491893')

cellranger count --create-bam=false -